# Notebook 08: MLP training on bike rentals by hand

In this notebook, we will train a small multilayer perceptron (MLP) to predict bike-rental counts from a curated real dataset.

The goal is not to use a machine-learning library. The goal is to see the whole training path with plain Python lists and floats: data row -> prediction -> loss -> gradients -> parameter update.

We will start by loading one row, building one-hot hour inputs, and comparing the scaled target with the original rental count. Then we will build the forward pass, derive the backward pass by hand, and train with plain gradient descent.


## First example: one row becomes model inputs and a target

The CSV file has one bike-rental example per row. A **model input** is a number the model is allowed to use for its prediction. A **target** is the answer the model is trying to predict.

For the first row, we will use 28 inputs: 24 one-hot hour values plus `temperature`, `humidity`, `wind_speed`, and `working_day`. The scaled target is `target_scaled`, which is the rental count divided by `1000`.


In [35]:
import csv

from pathlib import Path

DATA_PATH = "../data/bike_rentals_mini.csv"

rows = []

with Path(DATA_PATH).open(newline="") as csv_file:
    reader = csv.DictReader(csv_file)
    column_names = reader.fieldnames

    for row in reader:
        rows.append(row)


def row_to_input_values(row: dict[str, str]) -> list[float]:
    hour = int(row["hour"])
    hour_one_hot = [0.0 for _ in range(24)]
    hour_one_hot[hour] = 1.0

    return hour_one_hot + [
        float(row["temperature"]),
        float(row["humidity"]),
        float(row["wind_speed"]),
        float(row["working_day"]),
    ]


first_row = rows[0]
input_values = row_to_input_values(first_row)
target_scaled = float(first_row["target_scaled"])

print("columns:", column_names)
print("number of rows:", len(rows))
print("number of inputs:", len(input_values))
print("hour one-hot inputs:", input_values[:24])
print("weather/working-day inputs:", input_values[24:])
print("target_scaled:", target_scaled)
print("original rental_count:", first_row["rental_count"])


columns: ['hour', 'hour_scaled', 'temperature', 'humidity', 'wind_speed', 'working_day', 'rental_count', 'target_scaled']
number of rows: 240
number of inputs: 28
hour one-hot inputs: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
weather/working-day inputs: [0.3, 0.49, 0.2537, 1.0]
target_scaled: 0.083
original rental_count: 83


## Feature signal check

Before changing the model again, we should check whether the chosen inputs contain useful signal for `rental_count`.

Correlation gives a quick linear check for each feature. Grouped averages and simple baselines are more helpful for features like hour, where the pattern may rise and fall during the day instead of moving in one straight line.


In [36]:
def mean(values: list[float]) -> float:
    return sum(values) / len(values)


def root_mean_squared_error(errors: list[float]) -> float:
    squared_errors = [error * error for error in errors]
    return mean(squared_errors) ** 0.5


def pearson_correlation(xs: list[float], ys: list[float]) -> float:
    x_mean = mean(xs)
    y_mean = mean(ys)

    numerator = sum(
        (x - x_mean) * (y - y_mean)
        for x, y in zip(xs, ys)
    )
    x_denominator = sum((x - x_mean) ** 2 for x in xs) ** 0.5
    y_denominator = sum((y - y_mean) ** 2 for y in ys) ** 0.5

    if x_denominator == 0.0 or y_denominator == 0.0:
        return 0.0

    return numerator / (x_denominator * y_denominator)


def print_regression_metrics(
    label: str,
    predicted_counts: list[float],
    actual_counts: list[float],
) -> None:
    errors = [
        predicted_count - actual_count
        for predicted_count, actual_count in zip(predicted_counts, actual_counts)
    ]
    absolute_errors = [abs(error) for error in errors]

    within_50 = sum(1 for error in absolute_errors if error <= 50.0)
    within_100 = sum(1 for error in absolute_errors if error <= 100.0)

    print(label)
    print(f"  mean absolute error: {mean(absolute_errors):.1f} rentals")
    print(f"  root mean squared error: {root_mean_squared_error(errors):.1f} rentals")
    print(f"  within 50 rentals: {within_50 / len(actual_counts) * 100.0:.1f}%")
    print(f"  within 100 rentals: {within_100 / len(actual_counts) * 100.0:.1f}%")


actual_counts = [float(row["rental_count"]) for row in rows]
feature_names = [
    "hour_scaled",
    "temperature",
    "humidity",
    "wind_speed",
    "working_day",
]

print("Feature correlation with rental_count")
for feature_name in feature_names:
    feature_values = [float(row[feature_name]) for row in rows]
    correlation = pearson_correlation(feature_values, actual_counts)
    print(f"{feature_name}: {correlation:.3f}")

print()
print("Average rentals by hour")
for hour in range(24):
    hour_counts = [
        float(row["rental_count"])
        for row in rows
        if int(row["hour"]) == hour
    ]
    print(f"{hour:02d}: {mean(hour_counts):.1f}")

print()
print("Average rentals by working_day")
for working_day in [0, 1]:
    working_day_counts = [
        float(row["rental_count"])
        for row in rows
        if int(row["working_day"]) == working_day
    ]
    print(
        f"{working_day}: {mean(working_day_counts):.1f} "
        f"over {len(working_day_counts)} rows"
    )

print()
print("Simple baselines")

mean_count = mean(actual_counts)
mean_baseline_predictions = [mean_count for _ in rows]
print_regression_metrics(
    "Predict the overall average",
    mean_baseline_predictions,
    actual_counts,
)

hour_average_counts = {}
for hour in range(24):
    hour_counts = [
        float(row["rental_count"])
        for row in rows
        if int(row["hour"]) == hour
    ]
    hour_average_counts[hour] = mean(hour_counts)

hour_baseline_predictions = [
    hour_average_counts[int(row["hour"])]
    for row in rows
]
print_regression_metrics(
    "Predict the average for that hour",
    hour_baseline_predictions,
    actual_counts,
)


Feature correlation with rental_count
hour_scaled: 0.407
temperature: 0.399
humidity: -0.385
wind_speed: 0.087
working_day: 0.041

Average rentals by hour
00: 26.7
01: 42.9
02: 16.5
03: 7.4
04: 7.7
05: 25.5
06: 95.2
07: 253.4
08: 413.4
09: 172.4
10: 132.8
11: 139.9
12: 275.1
13: 216.6
14: 275.2
15: 188.1
16: 332.4
17: 519.1
18: 474.8
19: 330.4
20: 183.5
21: 206.0
22: 149.3
23: 89.1

Average rentals by working_day
0: 177.9 over 66 rows
1: 195.4 over 174 rows

Simple baselines
Predict the overall average
  mean absolute error: 148.2 rentals
  root mean squared error: 189.6 rentals
  within 50 rentals: 20.0%
  within 100 rentals: 37.9%
Predict the average for that hour
  mean absolute error: 83.2 rentals
  root mean squared error: 123.2 rentals
  within 50 rentals: 51.2%
  within 100 rentals: 68.3%


## Model shape

Each training example gives the model 28 input numbers:

1. 24 one-hot hour values, one for each hour from `0` to `23`
2. `temperature`
3. `humidity`
4. `wind_speed`
5. `working_day`

The model will use this shape:

```text
28 inputs -> 24 ReLU hidden neurons -> 1 output
```

The hidden size is configurable with `HIDDEN_SIZE`. More hidden neurons give the model more capacity, which means more small calculators can learn different patterns in the data.

Each hidden neuron produces one hidden value. The output layer combines those hidden values into one scaled rental-count prediction.


## Forward pass as a reusable function

A forward pass sends one example through the model to produce one prediction. We will put the forward pass inside a function right away, because training needs to reuse the same calculation for many rows.

Inside the function, Layer 1 turns the input values into hidden ReLU values. ReLU means “keep positive numbers, turn negative numbers into `0`.”

Layer 2 combines the hidden values into one scaled rental-count prediction. The function also returns a `ForwardCache` dataclass, which stores intermediate values the backward pass will need when it computes gradients.


In [37]:
import random
from dataclasses import dataclass


def relu(value: float) -> float:
    return max(0.0, value)


@dataclass
class ForwardCache:
    input_values: list[float]
    hidden_raw_values: list[float]
    hidden_outputs: list[float]


HIDDEN_SIZE = 24
INITIALIZATION_SEED = 7
WEIGHT_SCALE = 0.1


def make_initial_parameters(
    input_size: int,
    hidden_size: int,
    seed: int,
    weight_scale: float,
) -> tuple[list[list[float]], list[float], list[float], float]:
    rng = random.Random(seed)

    # Parameters for the input -> hidden layer
    input_to_hidden_weights = [
        [rng.uniform(-weight_scale, weight_scale) for _ in range(input_size)]
        for _ in range(hidden_size)
    ]
    hidden_biases = [0.0 for _ in range(hidden_size)]

    # Parameters for the hidden -> output layer
    hidden_to_output_weights = [
        rng.uniform(-weight_scale, weight_scale) for _ in range(hidden_size)
    ]
    output_bias = 0.0

    return (
        input_to_hidden_weights,
        hidden_biases,
        hidden_to_output_weights,
        output_bias,
    )


(
    input_to_hidden_weights,
    hidden_biases,
    hidden_to_output_weights,
    output_bias,
) = make_initial_parameters(
    input_size=len(input_values),
    hidden_size=HIDDEN_SIZE,
    seed=INITIALIZATION_SEED,
    weight_scale=WEIGHT_SCALE,
)


def forward_pass(
    input_values: list[float],
    input_to_hidden_weights: list[list[float]],
    hidden_biases: list[float],
    hidden_to_output_weights: list[float],
    output_bias: float,
) -> tuple[float, ForwardCache]:
    hidden_raw_values = []
    hidden_outputs = []

    # Layer 1: input values -> hidden ReLU values
    for hidden_index in range(len(hidden_biases)):
        raw_value = hidden_biases[hidden_index]

        for input_index in range(len(input_values)):
            raw_value += (
                input_values[input_index]
                * input_to_hidden_weights[hidden_index][input_index]
            )

        hidden_output = relu(raw_value)

        hidden_raw_values.append(raw_value)
        hidden_outputs.append(hidden_output)

    # Layer 2: hidden ReLU values -> one scaled prediction
    prediction = output_bias

    for hidden_index in range(len(hidden_outputs)):
        prediction += (
            hidden_outputs[hidden_index] * hidden_to_output_weights[hidden_index]
        )

    cache = ForwardCache(
        input_values=input_values,
        hidden_raw_values=hidden_raw_values,
        hidden_outputs=hidden_outputs,
    )

    return prediction, cache


prediction, cache = forward_pass(
    input_values,
    input_to_hidden_weights,
    hidden_biases,
    hidden_to_output_weights,
    output_bias,
)

print("hidden size:", len(hidden_biases))
print("first hidden neuron weights:", input_to_hidden_weights[0])
print("prediction_scaled:", prediction)
print("target_scaled:", target_scaled)
print("cached hidden raw values:", cache.hidden_raw_values)
print("cached hidden outputs:", cache.hidden_outputs)


hidden size: 24
first hidden neuron weights: [-0.03523344703336753, -0.06983016521509962, 0.03018689460797075, -0.08551274266649145, 0.007176400861337834, -0.026862216617482892, -0.08840021504505864, 0.0014871466378840459, -0.09250086831160304, -0.013270863267522831, -0.08602891528507622, -0.08185739733122699, -0.015096162171497202, 0.06537042493440762, -0.07523960777007088, -0.055352207078597095, 0.02548664448111787, 0.08954178849140113, 0.01542058972349973, -0.020663905069843974, 0.09525102111858402, -0.09068346387644875, 0.07169369180973592, -0.04207814273366475, -0.0711489833285125, -0.07644155238432633, -0.03830363517961313, 0.06322527182400628]
prediction_scaled: -0.00854677630394311
target_scaled: 0.083
cached hidden raw values: [0.020193228393182654, -0.05075920206722622, 0.0032520064305110036, -0.0050108798366770985, -0.10267971494534883, 0.07277163878830624, 0.13049077125731418, 0.046870231111367, -0.00550401867820564, -0.003841982286280722, 0.05968622443395167, -0.0612339988

## Backward pass as one reusable function

The backward pass starts at the prediction error and computes every gradient we need for this one example.

It walks backward through the same path the forward pass used:

```text
loss -> prediction -> output layer -> hidden ReLU values -> input-to-hidden weights
```

The function returns a `BackwardPassResult` dataclass. That dataclass only keeps the update-ready parameter gradients: `hidden_to_output_weights`, `output_bias`, `input_to_hidden_weights`, and `hidden_biases`.

Intermediate gradients such as `d_loss_d_prediction` and `d_loss_d_hidden_outputs` still exist inside the function, but they are not returned because gradient descent will not update those values directly.


In [38]:
from dataclasses import dataclass


@dataclass
class BackwardPassResult:
    d_loss_d_hidden_to_output_weights: list[float]
    d_loss_d_output_bias: float
    d_loss_d_input_to_hidden_weights: list[list[float]]
    d_loss_d_hidden_biases: list[float]


def relu_derivative(raw_value: float) -> float:
    if raw_value > 0.0:
        return 1.0

    return 0.0


def backward_pass(
    prediction: float,
    target: float,
    cache: ForwardCache,
    hidden_to_output_weights: list[float],
) -> BackwardPassResult:
    input_values = cache.input_values
    hidden_raw_values = cache.hidden_raw_values
    hidden_outputs = cache.hidden_outputs

    # Start of backprop: loss -> prediction
    error = prediction - target
    d_loss_d_prediction = 2.0 * error

    d_loss_d_hidden_to_output_weights = []
    d_loss_d_hidden_outputs = []

    # Output layer: prediction -> hidden_to_output weights and hidden outputs
    for hidden_index, hidden_output in enumerate(hidden_outputs):
        # Compute the derivative of the loss with respect to this hidden-to-output weight
        d_prediction_d_hidden_to_output_weight = hidden_output
        d_loss_d_hidden_to_output_weight = (
            d_loss_d_prediction * d_prediction_d_hidden_to_output_weight
        )
        d_loss_d_hidden_to_output_weights.append(d_loss_d_hidden_to_output_weight)

        # Compute the derivative of the loss with respect to this hidden output after ReLU
        d_prediction_d_hidden_output = hidden_to_output_weights[hidden_index]
        d_loss_d_hidden_output = d_loss_d_prediction * d_prediction_d_hidden_output
        d_loss_d_hidden_outputs.append(d_loss_d_hidden_output)

    d_loss_d_output_bias = d_loss_d_prediction

    d_loss_d_input_to_hidden_weights = []
    d_loss_d_hidden_biases = []

    # Hidden layer: hidden ReLU values -> hidden raw values -> input weights and biases
    for hidden_index, hidden_raw_value in enumerate(hidden_raw_values):
        d_hidden_output_d_hidden_raw = relu_derivative(hidden_raw_value)
        d_loss_d_hidden_raw = (
            d_loss_d_hidden_outputs[hidden_index] * d_hidden_output_d_hidden_raw
        )

        hidden_weight_gradients = []

        for input_value in input_values:
            d_hidden_raw_d_input_to_hidden_weight = input_value
            d_loss_d_input_to_hidden_weight = (
                d_loss_d_hidden_raw * d_hidden_raw_d_input_to_hidden_weight
            )
            hidden_weight_gradients.append(d_loss_d_input_to_hidden_weight)

        d_loss_d_input_to_hidden_weights.append(hidden_weight_gradients)

        d_loss_d_hidden_bias = d_loss_d_hidden_raw
        d_loss_d_hidden_biases.append(d_loss_d_hidden_bias)

    return BackwardPassResult(
        d_loss_d_hidden_to_output_weights=d_loss_d_hidden_to_output_weights,
        d_loss_d_output_bias=d_loss_d_output_bias,
        d_loss_d_input_to_hidden_weights=d_loss_d_input_to_hidden_weights,
        d_loss_d_hidden_biases=d_loss_d_hidden_biases,
    )


gradients = backward_pass(
    prediction,
    target_scaled,
    cache,
    hidden_to_output_weights,
)

current_error = prediction - target_scaled
current_loss = current_error * current_error

print("loss:", current_loss)
print("d_loss_d_hidden_to_output_weights:", gradients.d_loss_d_hidden_to_output_weights)
print("d_loss_d_output_bias:", gradients.d_loss_d_output_bias)
print(
    "d_loss_d_input_to_hidden_weights[0]:",
    gradients.d_loss_d_input_to_hidden_weights[0],
)
print("d_loss_d_hidden_biases:", gradients.d_loss_d_hidden_biases)

loss: 0.008380812251644202
d_loss_d_hidden_to_output_weights: [-0.0036972499251302503, -0.0, -0.0005954214104659508, -0.0, -0.0, -0.013324017874848843, -0.023892018892044703, -0.00858163712573286, -0.0, -0.0, -0.010928162873363834, -0.0, -0.0, -0.006984252033977093, -0.012729358821688639, -0.009496210168592638, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0]
d_loss_d_output_bias: -0.18309355260788623
d_loss_d_input_to_hidden_weights[0]: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.014203200960358818, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.004260960288107645, 0.006959568470575821, 0.003603352083643032, 0.014203200960358818]
d_loss_d_hidden_biases: [0.014203200960358818, 0.0, -0.0008948397858051369, -0.0, 0.0, 0.010122032875486499, -0.003700719736705608, 0.017926263497997606, 0.0, 0.0, -0.016805789977665533, -0.0, -0.0, 0.0009043276164511971, 0.009712450287744853, 0.009262395802763994, -0.0, -0.0, 0.0, 0.0, 0.0, -0.0, 0.0, 0.0]


## Training loop on all rows

A training loop repeats the same learning steps many times:

```text
forward pass -> backward pass -> average gradients -> parameter update
```

Now we will train on every row loaded from the full bike-rental CSV. For each epoch, we compute gradients for every row, average those gradients, and then apply one parameter update.

Averaging keeps the update size consistent when the number of training examples changes.


In [39]:
TrainingExample = tuple[list[float], float]


def row_to_training_example(row: dict[str, str]) -> TrainingExample:
    input_values = row_to_input_values(row)
    target_scaled = float(row["target_scaled"])

    return input_values, target_scaled


def zero_gradients(hidden_size: int, input_size: int) -> BackwardPassResult:
    return BackwardPassResult(
        d_loss_d_hidden_to_output_weights=[0.0 for _ in range(hidden_size)],
        d_loss_d_output_bias=0.0,
        d_loss_d_input_to_hidden_weights=[
            [0.0 for _ in range(input_size)] for _ in range(hidden_size)
        ],
        d_loss_d_hidden_biases=[0.0 for _ in range(hidden_size)],
    )


def add_gradients_in_place(
    total_gradients: BackwardPassResult,
    gradients: BackwardPassResult,
) -> None:
    for hidden_index, weight_gradient in enumerate(
        gradients.d_loss_d_hidden_to_output_weights
    ):
        total_gradients.d_loss_d_hidden_to_output_weights[hidden_index] += (
            weight_gradient
        )

    total_gradients.d_loss_d_output_bias += gradients.d_loss_d_output_bias

    for hidden_index, hidden_weight_gradients in enumerate(
        gradients.d_loss_d_input_to_hidden_weights
    ):
        for input_index, weight_gradient in enumerate(hidden_weight_gradients):
            total_gradients.d_loss_d_input_to_hidden_weights[hidden_index][
                input_index
            ] += weight_gradient

    for hidden_index, bias_gradient in enumerate(gradients.d_loss_d_hidden_biases):
        total_gradients.d_loss_d_hidden_biases[hidden_index] += bias_gradient


def scale_gradients(
    gradients: BackwardPassResult,
    scale: float,
) -> BackwardPassResult:
    return BackwardPassResult(
        d_loss_d_hidden_to_output_weights=[
            gradient * scale
            for gradient in gradients.d_loss_d_hidden_to_output_weights
        ],
        d_loss_d_output_bias=gradients.d_loss_d_output_bias * scale,
        d_loss_d_input_to_hidden_weights=[
            [gradient * scale for gradient in hidden_weight_gradients]
            for hidden_weight_gradients in gradients.d_loss_d_input_to_hidden_weights
        ],
        d_loss_d_hidden_biases=[
            gradient * scale for gradient in gradients.d_loss_d_hidden_biases
        ],
    )


def apply_gradients(
    input_to_hidden_weights: list[list[float]],
    hidden_biases: list[float],
    hidden_to_output_weights: list[float],
    output_bias: float,
    gradients: BackwardPassResult,
    learning_rate: float,
) -> float:
    # Update hidden -> output weights
    for hidden_index, weight_gradient in enumerate(
        gradients.d_loss_d_hidden_to_output_weights
    ):
        hidden_to_output_weights[hidden_index] -= learning_rate * weight_gradient

    # Update output bias
    output_bias -= learning_rate * gradients.d_loss_d_output_bias

    # Update input -> hidden weights
    for hidden_index, hidden_weight_gradients in enumerate(
        gradients.d_loss_d_input_to_hidden_weights
    ):
        for input_index, weight_gradient in enumerate(hidden_weight_gradients):
            input_to_hidden_weights[hidden_index][input_index] -= (
                learning_rate * weight_gradient
            )

    # Update hidden biases
    for hidden_index, bias_gradient in enumerate(gradients.d_loss_d_hidden_biases):
        hidden_biases[hidden_index] -= learning_rate * bias_gradient

    return output_bias


def mean_squared_error(
    training_examples: list[TrainingExample],
    input_to_hidden_weights: list[list[float]],
    hidden_biases: list[float],
    hidden_to_output_weights: list[float],
    output_bias: float,
) -> float:
    total_loss = 0.0

    for example_inputs, example_target in training_examples:
        prediction, _ = forward_pass(
            example_inputs,
            input_to_hidden_weights,
            hidden_biases,
            hidden_to_output_weights,
            output_bias,
        )
        error = prediction - example_target
        total_loss += error * error

    return total_loss / len(training_examples)


training_examples = [row_to_training_example(row) for row in rows]

(
    input_to_hidden_weights,
    hidden_biases,
    hidden_to_output_weights,
    output_bias,
) = make_initial_parameters(
    input_size=len(training_examples[0][0]),
    hidden_size=HIDDEN_SIZE,
    seed=INITIALIZATION_SEED,
    weight_scale=WEIGHT_SCALE,
)

learning_rate = 0.2
epochs = 500

initial_loss = mean_squared_error(
    training_examples,
    input_to_hidden_weights,
    hidden_biases,
    hidden_to_output_weights,
    output_bias,
)
print(f"initial training loss: {initial_loss:.6f}")

for epoch in range(epochs):
    total_gradients = zero_gradients(
        hidden_size=len(hidden_biases),
        input_size=len(input_to_hidden_weights[0]),
    )
    total_loss = 0.0

    for example_inputs, example_target in training_examples:
        prediction, cache = forward_pass(
            example_inputs,
            input_to_hidden_weights,
            hidden_biases,
            hidden_to_output_weights,
            output_bias,
        )
        error = prediction - example_target
        total_loss += error * error

        gradients = backward_pass(
            prediction,
            example_target,
            cache,
            hidden_to_output_weights,
        )
        add_gradients_in_place(total_gradients, gradients)

    average_loss = total_loss / len(training_examples)
    average_gradients = scale_gradients(
        total_gradients,
        1.0 / len(training_examples),
    )

    output_bias = apply_gradients(
        input_to_hidden_weights,
        hidden_biases,
        hidden_to_output_weights,
        output_bias,
        average_gradients,
        learning_rate,
    )

    if epoch == 0 or (epoch + 1) % 50 == 0:
        print(f"epoch: {epoch + 1} training loss: {average_loss:.6f}")

final_loss = mean_squared_error(
    training_examples,
    input_to_hidden_weights,
    hidden_biases,
    hidden_to_output_weights,
    output_bias,
)

print(f"final training loss: {final_loss:.6f}")

for row_index, (example_inputs, example_target) in enumerate(
    training_examples[:5],
):
    prediction, _ = forward_pass(
        example_inputs,
        input_to_hidden_weights,
        hidden_biases,
        hidden_to_output_weights,
        output_bias,
    )
    predicted_count = prediction * 1000.0
    actual_count = example_target * 1000.0

    print(
        f"row: {row_index + 1} "
        f"predicted rentals: {predicted_count:.1f} "
        f"actual rentals: {actual_count:.1f}"
    )


initial training loss: 0.075387
epoch: 1 training loss: 0.075387
epoch: 50 training loss: 0.031914
epoch: 100 training loss: 0.023696
epoch: 150 training loss: 0.015364
epoch: 200 training loss: 0.012196
epoch: 250 training loss: 0.011327
epoch: 300 training loss: 0.010868
epoch: 350 training loss: 0.010512
epoch: 400 training loss: 0.010209
epoch: 450 training loss: 0.009930
epoch: 500 training loss: 0.009675
final training loss: 0.009670
row: 1 predicted rentals: 246.2 actual rentals: 83.0
row: 2 predicted rentals: 170.0 actual rentals: 89.0
row: 3 predicted rentals: 73.2 actual rentals: 36.0
row: 4 predicted rentals: 112.7 actual rentals: 139.0
row: 5 predicted rentals: 62.2 actual rentals: 53.0


In [40]:
# For regression, "accuracy" needs a tolerance.
# Here we report average error and the percentage of predictions
# that land within 50 and 100 rentals of the actual count.

total_absolute_error = 0.0
total_squared_error = 0.0
within_50_rentals = 0
within_100_rentals = 0

for example_inputs, example_target in training_examples:
    prediction, _ = forward_pass(
        example_inputs,
        input_to_hidden_weights,
        hidden_biases,
        hidden_to_output_weights,
        output_bias,
    )

    predicted_count = prediction * 1000.0
    actual_count = example_target * 1000.0
    absolute_error = abs(predicted_count - actual_count)

    total_absolute_error += absolute_error
    total_squared_error += absolute_error * absolute_error

    if absolute_error <= 50.0:
        within_50_rentals += 1

    if absolute_error <= 100.0:
        within_100_rentals += 1

example_count = len(training_examples)
mean_absolute_error = total_absolute_error / example_count
root_mean_squared_error = (total_squared_error / example_count) ** 0.5
within_50_accuracy = within_50_rentals / example_count * 100.0
within_100_accuracy = within_100_rentals / example_count * 100.0

print(f"mean absolute error: {mean_absolute_error:.1f} rentals")
print(f"root mean squared error: {root_mean_squared_error:.1f} rentals")
print(f"within 50 rentals: {within_50_accuracy:.1f}%")
print(f"within 100 rentals: {within_100_accuracy:.1f}%")


mean absolute error: 72.8 rentals
root mean squared error: 98.3 rentals
within 50 rentals: 45.8%
within 100 rentals: 77.5%
